# Snake classification

### Import libraries

In [ ]:
!pip install ydata-profiling
!pip install  "numpy<2.4"

In [ ]:
import io
import os
import zipfile
from pathlib import PurePosixPath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from tensorflow.keras.applications import EfficientNetV2L
from tensorflow.keras.callbacks import EarlyStopping
from ydata_profiling import ProfileReport


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Dataset paths and metadata loading into variables

In [ ]:
train_zip_path = "/content/drive/MyDrive/amm_hazi/train_images_small.zip"
test_zip_path = "/content/drive/MyDrive/amm_hazi/test_images_small.zip"
csv_path = "/content/drive/MyDrive/amm_hazi/train_images_metadata.csv"
venomous_path = "/content/drive/MyDrive/amm_hazi/venomous_status_metadata.csv"

venomous_df = pd.read_csv(venomous_path)
df = pd.read_csv(csv_path)

df = df.merge(venomous_df[["class_id", "MIVS"]], on="class_id", how="left")

### Extracting to the colab server

In [ ]:
# Kicsomagolás a Colab gyors, lokális SSD-jére

local_train_dir = "/content/train_images"

# Csak akkor csomagoljuk ki, ha még nincs ott
if not os.path.exists(local_train_dir):
    os.makedirs(local_train_dir)
    print("Extracting files...")
    with zipfile.ZipFile(train_zip_path, "r") as zip_ref:
        zip_ref.extractall(local_train_dir)
    print("Extracting done!")
else:
    print("The piuctures are already extracted.")


### ZIP Indexing and Filtering Missing Images

> Only needed if reading from locally stored ZIP

In [ ]:
train_zip_map = {}

with zipfile.ZipFile(train_zip_path, "r") as z:
    for internal_path in z.namelist():
        if internal_path.lower().endswith(
            (".jpg", ".jpeg")
        ):
            filename = PurePosixPath(internal_path).name
            train_zip_map[filename] = internal_path

# 3. Filter out rows from dataframe that only exist in the metadata but not in train_images_small.zip
print(f"Total rows in CSV: {len(df)}")
df = df[df["image_path"].isin(train_zip_map)].reset_index(drop=True)
print(f"Matched images found in zip: {len(df)}")


### First five rows of ther dataframe

In [ ]:
df[:5]

### Number of classes

In [ ]:
snake_classes = df["class_id"].nunique()
snake_classes

### Creating a data profile report

In [ ]:
profile = ProfileReport(
    df,
    title="Snake Species Classification Profiling Report",
    vars={
        "num": {"chi_squared_threshold": 0.0}
    },  # Outlierek miatt sok memória fogyasztás
)

profile.to_file("snake_species_classification_profiling_report.html")


### Loss and accuracy visualization

In [ ]:
def accuracy_loss_plots(history, suptitle):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Plot training & validation accuracy
    ax1.plot(history.history["accuracy"], label="Train")
    ax1.plot(history.history["val_accuracy"], label="Validation")
    ax1.set_title("Model Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend(loc="lower right")
    ax1.grid(True)

    # Plot training & validation loss
    ax2.plot(history.history["loss"], label="Train")
    ax2.plot(history.history["val_loss"], label="Validation")
    ax2.set_title("Model Loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend(loc="upper right")
    ax2.grid(True)

    plt.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

### Model evaluation

In [ ]:
def evaluate_model(model, val_df, val_loader):
    # Minden validációs képre megkapjuk a fajok valószínűségi értékeit
    val_preds_prob = model.predict(val_loader)
    # A legngyobb érték indexe (a jósolt kigyófaj)
    val_preds = np.argmax(val_preds_prob, axis=1)

    # A valódi kígyófaj indexe
    val_true = val_df["model_label"].values
    # A valódi mérgességi mutató
    val_venomous_true = val_df["MIVS"].astype(bool).values

    # Szótár, mely megmutatja, melyik faj mérges
    val_venomous_pred = np.array(
        [label_to_mivs[p] for p in val_preds],
        dtype=int
        )

    # Pontosság
    accuracy = accuracy_score(val_true, val_preds)

    # Macro F1
    macro_f1 = f1_score(val_true, val_preds, average="macro", zero_division = 0)

    # Metric 3: Venomous Binary Accuracy (helyes bináris döntés / összes bináris döntés)
    venomous_binary_acc = accuracy_score(val_venomous_true, val_venomous_pred)

    # Metric 4: Venomous Error-Weighted Accuracy (Custom penalty weighting)
    losses = []
    for t_cls, p_cls, t_ven, p_ven in zip(
        val_true, val_preds, val_venomous_true, val_venomous_pred
    ):
        if t_cls == p_cls:
            losses.append(0.0)
        elif t_ven == 0 and p_ven == 0:
            losses.append(1.0)
        elif t_ven == 0 and p_ven == 1 or t_ven == 1 and p_ven == 1:
            losses.append(2.0)
        elif t_ven == 1 and p_ven == 0:
            losses.append(5.0)

    average_loss = np.mean(losses)

    return accuracy, macro_f1, venomous_binary_acc, average_loss

### Data loader class

> Only needed if reading from locally stored ZIP

In [ ]:
class ZipSnakeDataset(keras.utils.Sequence):
    def __init__(
        self,
        dataframe,
        zip_path,
        zip_map,
        batch_size=32,
        img_size=(224, 224),
        shuffle=True,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.df = dataframe
        self.zip_path = zip_path
        self.zip_map = zip_map
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.zip_handle = None
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        # Open zip file safely per worker
        if self.zip_handle is None:
            self.zip_handle = zipfile.ZipFile(self.zip_path, "r")

        # Get batch row indices
        batch_indices = self.indices[
            idx * self.batch_size : (idx + 1) * self.batch_size
        ]
        batch_rows = self.df.iloc[batch_indices]

        images, labels = [], []
        for _, row in batch_rows.iterrows():
            # Lookup the exact internal zip path using your train_zip_map
            internal_path = self.zip_map[row["image_path"]]
            raw_bytes = self.zip_handle.read(internal_path)

            # Decode and resize to 224x224
            img = Image.open(io.BytesIO(raw_bytes)).convert("RGB").resize(self.img_size)
            images.append(np.array(img, dtype=np.float32))
            labels.append(row["model_label"])

        return np.array(images), np.array(labels, dtype=np.int32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


### Train-validation split

In [ ]:
local_train_dir = "/content/train_images"
# Hozzáadjuk az almappa nevét az útvonalhoz:
image_dir = os.path.join(local_train_dir, "train_images_small")

# Most már az image_dir-t használjuk az elérési utak generálásához:
df["full_image_path"] = df["image_path"].apply(lambda x: os.path.join(image_dir, x))

# Ellenőrzés és szűrés
df = df[df["full_image_path"].apply(os.path.exists)].reset_index(drop=True)
print(f"Meglévő, szűrt képek száma: {len(df)}")

In [ ]:
df["model_label"], unique_classes = pd.factorize(df["class_id"])

label_to_mivs = (
    df.drop_duplicates("model_label")
      .set_index("model_label")["MIVS"]
      .to_dict()
)

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {
    "accuracy": [],
    "macro_f1": [],
    "venomous_accuracy": [],
    "weighted_cost": []
}

In [ ]:
# 1. Képbetöltő és előfeldolgozó függvény
def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    # Átméretezés az EfficientNetV2L 480x480-as méretére (vagy 224-re, ha a custom CNN-t használod)
    image = tf.image.resize(image, [384, 384])

    return image, label


# 2. Adatkészlet (Dataset) létrehozó függvény
def create_dataset(dataframe, batch_size=64, is_training=True):
    # Elérési utak és címkék kinyerése a már korábban jól előkészített dataframe-ből
    paths = dataframe["full_image_path"].values
    labels = dataframe["model_label"].values

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if is_training:
        dataset = dataset.shuffle(buffer_size=len(dataframe))

    AUTOTUNE = tf.data.AUTOTUNE
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=AUTOTUNE)

    return dataset

### Data loader

> Only needed if reading from locally stored ZIP

In [ ]:
# Initialize the loaders (using the train_zip_map you already built)
train_loader = ZipSnakeDataset(
    train_df, train_zip_path, train_zip_map, batch_size=128, shuffle=True
)
val_loader = ZipSnakeDataset(
    val_df, train_zip_path, train_zip_map, batch_size=128, shuffle=False
)


### A few example images

In [ ]:
images, labels = train_loader[0]

random_indices = np.random.choice(images.shape[0], size=10, replace=False)

plt.figure(figsize=(10, 10))
for i, idx in enumerate(random_indices):
    ax = plt.subplot(5, 5, i + 1)

    img = images[idx].astype(np.uint8)

    plt.imshow(img)
    plt.title(f"Class Label: {labels[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## Transfer learning

In [ ]:
mixed_precision.set_global_policy("mixed_float16")

print(mixed_precision.global_policy())

In [ ]:
# Alapmodell betöltése a "fej" nélkül (include_top=False)
def build_transfer_model(num_classes):
    # 1. Alapmodell betöltése a Keras-ból, 480x480 bemenettel, a "fej" (include_top) nélkül
    base_model = EfficientNetV2L(
        weights="imagenet", input_shape=(384, 384, 3), include_top=False
    )

    # 2. Bemelegítő fázis: alapmodell befagyasztása
    base_model.trainable = False

    # 3. Új hálózat felépítése
    inputs = keras.Input(shape=(384, 384, 3))

    # Adatbővítés (opcionális, de ajánlott 300 osztályhoz)
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(height_factor=(-0.15, 0.15), width_factor=(-0.15, 0.15))(x)
    # Futtatás az alapmodellen
    x = base_model(x, training=False)

    # Klasszifikációs fej
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = keras.Model(inputs, outputs)

    # A modell fordítása (Compile)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model, base_model

In [ ]:
class MacroF1Callback(keras.callbacks.Callback):
    def __init__(self, val_dataset, val_labels):
        super().__init__()
        self.val_dataset = val_dataset
        self.val_labels = val_labels

    def on_epoch_end(self, epoch, logs=None):

        # Predikciók a validációs adatokon
        predictions = self.model.predict(self.val_dataset, verbose=0)

        predicted_labels = np.argmax(predictions, axis=1)

        # Macro F1 kiszámítása
        macro_f1 = f1_score(
            self.val_labels,
            predicted_labels,
            average="macro",
            zero_division=0
        )

        # Hozzáadjuk a Keras naplójához
        logs["val_macro_f1"] = macro_f1

        print(f" - val_macro_f1: {macro_f1:.4f}")

In [ ]:
def fine_tuning(model, base_model, train_dataset, val_dataset):
  print("Finetuning phase...")

  # 1. Az alapmodell teljes feloldása
  base_model.trainable = True

  # for layer in base_model.layers[:-200]:
  #    layer.trainable = False

  # 2. Újrafordítás egy NAGYON ALACSONY tanulási rátával (pl. 1e-5)
  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=1e-5),
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"],
  )

  # 3. Korai leállítás (Early Stopping)
  early_stop = EarlyStopping(
      monitor="val_macro_f1", mode="max", patience=3, restore_best_weights=True, verbose=1
  )

  macro_f1_callback = MacroF1Callback(
      val_dataset=val_dataset,
      val_labels=val_df["model_label"].values
  )
  # 4. Továbbtanítás a már bemelegített állapotból
  history_finetune = model.fit(
      train_dataset,
      validation_data=val_dataset,
      epochs=50,
      callbacks=[early_stop, macro_f1_callback, reduce_lr_1],
  )

  return history_finetune

In [ ]:
warmup_early_stop = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=3,
    restore_best_weights=True,
    verbose=1
)
reduce_lr_1 = tf.keras.callbacks.ReduceLROnPlateau(
      monitor='val_loss',
      factor=0.5,      # Ha elakad, megfelezi a sebességet
      patience=2,      # 2 epocha stagnálás után lép
      min_lr=1e-5
)

In [ ]:
for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(df, df["model_label"], df["observation_id"]),
    start = 1
    ):

  train_df = df.iloc[train_idx].reset_index(drop=True)
  val_df = df.iloc[val_idx].reset_index(drop=True)

  train_dataset = create_dataset(
        train_df, batch_size=16, is_training=True
    )
  val_dataset = create_dataset(
      val_df, batch_size=16, is_training=False
    )


  model, base_model = build_transfer_model(snake_classes)

  # Első tanítási fázis
  history_warmup = model.fit(
      train_dataset,
      validation_data=val_dataset,
      epochs=10,
      callbacks=[warmup_early_stop, reduce_lr_1]
    )

  history_finetune = fine_tuning(
    model, base_model, train_dataset, val_dataset
  )
  accuracy, macro_f1, venomous_acc, weighted_cost = evaluate_model(
      model, val_df, val_dataset
  )

  results["accuracy"].append(accuracy)
  results["macro_f1"].append(macro_f1)
  results["venomous_accuracy"].append(venomous_acc)
  results["weighted_cost"].append(weighted_cost)

for metric, values in results.items():
    print(f"{metric}: {np.mean(values):.4f} ± {np.std(values):.4f}")

### Training the last layers for finetuning

### Evaluating the modell

In [ ]:
tl_model_evaluation = evaluate_model(model, val_df, val_dataset)

print(f"Accuracy: {tl_model_evaluation[0]:.4f}")
print(f"Macro F1 score: {tl_model_evaluation[1]:.4f}")
print(f"Venomous Binary Accuracy {tl_model_evaluation[2]:.4f}")
print(f"Venomous Error-Weighted Accuracy {tl_model_evaluation[3]:.4f}")

### Visualizing the accuracy and loss over time

In [ ]:
accuracy_loss_plots(history_warmup, "Warmup")
accuracy_loss_plots(history_finetune, "Finetune")

### Confusion data

In [ ]:
# 1. Predikciók elkészítése
val_probs = model.predict(val_dataset, verbose=1)
val_preds = np.argmax(val_probs, axis=1)

# 2. Valódi címkék
val_true = val_df["model_label"].values

# 3. Osztályok nevei (nálad ezek a class_id értékek)
class_names = [str(x) for x in unique_classes]

# 4. Konfúziós mátrix
cm = confusion_matrix(val_true, val_preds, labels=np.arange(len(class_names)))


In [ ]:
# A főátlót nullázzuk, mert a helyes felismerések nem érdekelnek most
cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)

# A leggyakoribb tévesztések helyei
top_errors = np.dstack(
    np.unravel_index(np.argsort(cm_errors.ravel())[::-1], cm_errors.shape)
)[0]

rows = []

for true_idx, pred_idx in top_errors[:30]:
    count = cm[true_idx, pred_idx]

    if count == 0:
        continue

    total_true = cm[true_idx].sum()
    error_rate = count / total_true

    rows.append(
        {
            "True species": class_names[true_idx],
            "Predicted species": class_names[pred_idx],
            "Misclassified images": count,
            "Error rate within true class": f"{error_rate:.1%}",
        }
    )

top_errors_df = pd.DataFrame(rows)

display(top_errors_df)

In [ ]:
num_classes = len(unique_classes)

# 3. Fajpárok összesített tévesztése
# A -> B és B -> A tévesztéseket összeadjuk
pair_errors = []

for i in range(num_classes):
    for j in range(i + 1, num_classes):
        errors = cm[i, j] + cm[j, i]

        if errors > 0:
            pair_errors.append((i, j, errors))

# Leggyakoribb 5 fajpár
pair_errors.sort(key=lambda x: x[2], reverse=True)
top_pairs = pair_errors[:5]

# 4. Képek megjelenítése
fig, axes = plt.subplots(len(top_pairs), 4, figsize=(16, 4 * len(top_pairs)))

if len(top_pairs) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, (class_a, class_b, total_errors) in enumerate(top_pairs):
    # A -> B: valódi A, jósolt B
    errors_a_to_b = val_df[(val_df["model_label"] == class_a) & (val_preds == class_b)]

    # B -> A: valódi B, jósolt A
    errors_b_to_a = val_df[(val_df["model_label"] == class_b) & (val_preds == class_a)]

    # Maximum 2-2 véletlenszerű kép kiválasztása
    samples_a = errors_a_to_b.sample(n=min(2, len(errors_a_to_b)), random_state=42)

    samples_b = errors_b_to_a.sample(n=min(2, len(errors_b_to_a)), random_state=42)

    samples = [(samples_a, class_a, class_b), (samples_b, class_b, class_a)]

    col = 0

    for sample_df, true_class, predicted_class in samples:
        for _, image_row in sample_df.iterrows():
            img = Image.open(image_row["full_image_path"]).convert("RGB")

            axes[row, col].imshow(img)

            axes[row, col].set_title(
                f"True: {unique_classes[true_class]}\n"
                f"Predicted: {unique_classes[predicted_class]}"
            )

            axes[row, col].axis("off")
            col += 1

    # Ha valamelyik irányban nincs 2 kép, a maradék tengelyeket elrejtjük
    while col < 4:
        axes[row, col].axis("off")
        col += 1

    fig.text(
        0.5,
        1 - (row + 0.5) / len(top_pairs),
        f"Total confusions: {total_errors}",
        ha="center",
        fontsize=11,
    )

plt.suptitle("TOP 5 Most Confused Species Pairs", fontsize=18, y=1.01)

plt.tight_layout()
plt.show()
